In [ ]:
# =============================
# INDICADOR DE SOBRE-ENDEUDAMIENTO
# =============================

# ---------------------------------
# 1. IMPORTAR LIBRERÍAS
# ---------------------------------

import pandas as pd
import numpy as np


# ---------------------------------
# 2. CARGAR ARCHIVO EXCEL
# ---------------------------------

# En Google Colab:
# 1. Ejecutar esta celda
# 2. Subir el archivo Excel cuando aparezca la ventana

from google.colab import files
uploaded = files.upload()


# Obtener nombre del archivo
archivo = list(uploaded.keys())[0]


# Leer Excel
# Saltamos las primeras filas porque contienen título

df = pd.read_excel(archivo, skiprows=2)


# Mostrar datos
print("\nDATOS CARGADOS:\n")
print(df)


# ---------------------------------
# 3. LIMPIEZA DE DATOS
# ---------------------------------

# Eliminar filas vacías

df = df.dropna(how='all')


# Renombrar columnas por seguridad

df.columns = [
    'numero_operacion',
    'usuario',
    'categoria',
    'descripcion',
    'monto',
    'cuotas',
    'tna'
]


# ---------------------------------
# 4. INGRESO MENSUAL DEL USUARIO
# ---------------------------------

# Simulación de ingreso mensual
# Puede modificarse libremente

ingreso_mensual = 1200000

print(f"\nIngreso mensual considerado: ${ingreso_mensual:,.0f}")


# ---------------------------------
# 5. CÁLCULO DE CUOTA MENSUAL
# ---------------------------------

# Conversión de TNA a tasa mensual

df['tasa_mensual'] = (df['tna'] / 100) / 12


# Fórmula francesa simplificada

def calcular_cuota(monto, tasa, cuotas):

    if tasa == 0:
        return monto / cuotas

    cuota = monto * (
        tasa * (1 + tasa)**cuotas
    ) / (
        ((1 + tasa)**cuotas) - 1)

    return cuota


# Aplicar cálculo

df['cuota_mensual'] = df.apply(
    lambda fila: calcular_cuota(
        fila['monto'],
        fila['tasa_mensual'],
        fila['cuotas']
    ),
    axis=1
)


# ---------------------------------
# 6. ENDEUDAMIENTO TOTAL
# ---------------------------------

# Suma de cuotas mensuales

total_cuotas = df['cuota_mensual'].sum()


# Relación deuda / ingreso

ratio_endeudamiento = total_cuotas / ingreso_mensual


print("\n============================")
print("ANÁLISIS FINANCIERO")
print("============================")

print(f"\nTotal de cuotas mensuales: ${total_cuotas:,.0f}")
print(f"Ratio de endeudamiento: {ratio_endeudamiento:.2%}")


# ---------------------------------
# 7. DETECCIÓN DE GASTOS IMPULSIVOS
# ---------------------------------

# Categorías consideradas impulsivas

categorias_impulsivas = [
    'Tecnología',
    'Entretenimiento',
    'Indumentaria',
    'Viajes'
]


# Filtrar gastos impulsivos

gastos_impulsivos = df[
    df['categoria'].isin(categorias_impulsivas)
]


# Total gastos impulsivos

total_impulsivos = gastos_impulsivos['monto'].sum()


# Porcentaje sobre ingresos

ratio_impulsivos = total_impulsivos / ingreso_mensual


print(f"\nGastos impulsivos detectados: ${total_impulsivos:,.0f}")
print(f"Impacto impulsivo sobre ingresos: {ratio_impulsivos:.2%}")


# ---------------------------------
# 8. INDICADOR DE RIESGO
# ---------------------------------

riesgo = 0


# Riesgo por endeudamiento

if ratio_endeudamiento >= 0.50:
    riesgo += 3
elif ratio_endeudamiento >= 0.35:
    riesgo += 2
elif ratio_endeudamiento >= 0.20:
    riesgo += 1


# Riesgo por gastos impulsivos

if ratio_impulsivos >= 1:
    riesgo += 3
elif ratio_impulsivos >= 0.50:
    riesgo += 2
elif ratio_impulsivos >= 0.25:
    riesgo += 1


# ---------------------------------
# 9. CLASIFICACIÓN FINAL
# ---------------------------------

print("\n============================")
print("RESULTADO FINAL")
print("============================")

if riesgo <= 1:
    nivel = "BAJO"
    alerta = "Situación financiera estable"

elif riesgo <= 3:
    nivel = "MEDIO"
    alerta = "Atención: el nivel de deuda comienza a ser elevado"

else:
    nivel = "ALTO"
    alerta = "ALERTA: posible sobreendeudamiento"


print(f"\nNivel de riesgo: {nivel}")
print(f"Mensaje: {alerta}")


# ---------------------------------
# 10. ALERTAS PERSONALIZADAS
# ---------------------------------

print("\n============================")
print("ALERTAS DETECTADAS")
print("============================")


if ratio_endeudamiento > 0.35:
    print("- Más del 35% de los ingresos se destinan a cuotas")

if len(gastos_impulsivos) >= 3:
    print("- Se detectaron múltiples compras impulsivas")

if df['cuotas'].max() >= 12:
    print("- Existen operaciones con financiamiento largo")

if df['tna'].max() >= 40:
    print("- Existen créditos con tasas elevadas")


# ---------------------------------
# 11. TABLA FINAL
# ---------------------------------

print("\n============================")
print("DETALLE DE OPERACIONES")
print("============================")

columnas_mostrar = [
    'categoria',
    'descripcion',
    'monto',
    'cuotas',
    'tna',
    'cuota_mensual'
]

print(df[columnas_mostrar])

Saving Ejemplo operaciones de Ana.xlsx to Ejemplo operaciones de Ana.xlsx

DATOS CARGADOS:

   Número de operación Usuario             Compra  \
0                    1     Ana         Tecnología   
1                    2     Ana       Indumentaria   
2                    3     Ana             Viajes   
3                    4     Ana              Hogar   
4                    5     Ana              Salud   
5                    6     Ana    Entretenimiento   
6                    7     Ana          Educación   
7                    8     Ana          Movilidad   
8                    9     Ana  Electrodomésticos   
9                   10     Ana               Ocio   

                                         Descripción    Monto  Cuotas  TNA  
0               Compra de celular Samsung Galaxy S25  1250000      12   42  
1         Zapatillas deportivas y ropa para gimnasio   185000       3   18  
2  Pasajes y reserva de hotel para escapada a Bar...   780000       9   36  
3               